# MEDDIAG — Run 4 (recall retrain, TRAINING ONLY)

Clone of the main notebook, configured to **train** Stage-2 with the recall-oriented loss and upload to `vlm-session-state-run4`. The eval notebook (`kaggle_train.ipynb`) is untouched.


In [ ]:
# 1. CONFIG (RUN 4 — recall-tuned retrain, TRAINING ONLY) =====================
# Separate training notebook. Retrains Stage-2 with the recall-oriented loss
# (up-weighted ABNORMAL + focal) so the model stops missing abnormals. Uploads to
# a NEW dataset so your Run-3 (vlm-session-state) stays completely untouched.
import os, re

IS_FIRST_SESSION = True           # True = train Run-4 FROM SCRATCH (clean comparison).
                                  # Flip to False on a RESUME session to continue Run-4.
MODE = "train"                    # this notebook only trains (never evaluates)

# ── Run-4 recall knobs (exported to the trainer via env in cell 3) ───────────
CLS_POS_WEIGHT  = 2.5             # up-weight ABNORMAL -> higher recall  (1.0 = Run-3)
CLS_FOCAL_GAMMA = 2.0             # focal loss on hard/missed abnormals  (0.0 = off)

# ── Run-4 AUROC levers (exported to the trainer via env in cell 3) ───────────
VISION_FINETUNE  = True           # unfreeze EfficientNet-B0 -> features adapt to CXR
VISION_LR        = 1e-5           # LR for the unfrozen encoder (~10x below LoRA)
VISION_FT_BLOCKS = 2              # last N encoder blocks to fine-tune (0 = all)
VISION_AUGMENT   = True           # train-time rotation / brightness / contrast

def _detect_kaggle_username():
    for root, _dirs, _files in os.walk("/kaggle/input"):
        m = re.search(r"/kaggle/input/datasets/([^/]+)/", root.rstrip("/") + "/")
        if m:
            return m.group(1)
    return None

KAGGLE_USERNAME   = _detect_kaggle_username() or "sujalprasad"
PROJECTOR_DATASET = "vlm-projector"
STATE_DATASET     = "vlm-session-state-run4"   # NEW — Run-3's vlm-session-state stays intact

MAX_PAIRS  = 4000
EPOCHS     = 3
GRAD_ACCUM = 4
SAVE_EVERY = 250
LR         = 2e-4
EVAL_SAMPLES = 200                # unused in train mode; kept so cell 8 never NameErrors

print("Config (RUN 4 - recall retrain, training only):")
print(f"  session        : {'FIRST (from scratch)' if IS_FIRST_SESSION else 'RESUME'}")
print(f"  kaggle user    : {KAGGLE_USERNAME}")
print(f"  cls_pos_weight : {CLS_POS_WEIGHT} | focal_gamma : {CLS_FOCAL_GAMMA}")
print(f"  vision_finetune: {VISION_FINETUNE} (lr={VISION_LR}, last {VISION_FT_BLOCKS} blocks) | augment: {VISION_AUGMENT}")
print(f"  upload target  : {STATE_DATASET}  (Run-3 dataset untouched)")
print(f"  grad_accum     : {GRAD_ACCUM} | max_pairs : {MAX_PAIRS} | epochs : {EPOCHS}")


In [ ]:
# 2. PACKAGES =================================================================
import subprocess, sys
def pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
pip("--upgrade", "kaggle")          # ensure latest CLI (older one had an upload bug)
pip("peft>=0.19.1")
pip("bitsandbytes>=0.49.2")
pip("accelerate>=1.13.0")
pip("faiss-cpu==1.13.2")
pip("sentence-transformers")
pip("python-dotenv")
print("Packages ready.")


In [ ]:
# 3. SECRETS & ENV ============================================================
import os, torch
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"]                 = secrets.get_secret("HF_TOKEN")
os.environ["SEMANTIC_SCHOLAR_API_KEY"] = secrets.get_secret("SEMANTIC_SCHOLAR_API_KEY")
os.environ["CUBLAS_WORKSPACE_CONFIG"]  = ":4096:8"
os.environ["MEDDIAG_MAX_VRAM_GB"]      = "14"   # full 16GB T4 (4GB hypothesis confirmed)
os.environ["MEDDIAG_MIMIC_REPO"]       = "power2004/mimic-cxr-dataset"   # replaces deleted itsanmolgupta mirror (same image+findings+impression, 30633 ex)
os.environ["MEDDIAG_CLS_POS_WEIGHT"]   = str(CLS_POS_WEIGHT)    # Run-4 recall loss
os.environ["MEDDIAG_CLS_FOCAL_GAMMA"]  = str(CLS_FOCAL_GAMMA)
os.environ["MEDDIAG_VISION_FINETUNE"]  = "1" if VISION_FINETUNE else "0"   # Run-4 AUROC levers
os.environ["MEDDIAG_VISION_LR"]        = str(VISION_LR)
os.environ["MEDDIAG_VISION_FT_BLOCKS"] = str(VISION_FT_BLOCKS)
os.environ["MEDDIAG_VISION_AUGMENT"]   = "1" if VISION_AUGMENT else "0"

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Secrets loaded, HF login OK. Full 16 GB mode.")


In [ ]:
# 4. CLONE REPO (fresh) =======================================================
import subprocess, os
REPO_URL = "https://github.com/Sreenjoyee/visual-language-model-research-qlora-cot-rag.git"
REPO_DIR = "/kaggle/working/vlm"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth=1", REPO_URL, REPO_DIR])
    print("Cloned ->", REPO_DIR)
else:
    subprocess.call(["git", "-C", REPO_DIR, "reset", "--hard"])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    print("Updated ->", REPO_DIR)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())


In [ ]:
# 5. RESTORE ASSETS ===========================================================
# Projector from attached input. Session-state DOWNLOADED FRESH via API (latest
# version). Then we WIPE any stale on-disk checkpoints and restore exactly what
# the dataset has -> defeats Kaggle persistence resurrecting old checkpoints, so
# resume always matches the dataset's highest checkpoint.
import shutil, re, os, json, subprocess
from pathlib import Path

MODELS_DIR = Path(REPO_DIR) / "models"; MODELS_DIR.mkdir(exist_ok=True)

def _walk_find(base, name, want_dir=False):
    for root, dirs, files in os.walk(base):
        pool = dirs if want_dir else files
        if name in pool:
            return Path(root) / name
    return None

# ── Projector (from attached vlm-projector input) ─────────────────────────────
proj = _walk_find("/kaggle/input", "projector_stage1.pt")
if proj is None:
    raise FileNotFoundError("projector_stage1.pt not found - attach the vlm-projector dataset")
shutil.copy2(proj, MODELS_DIR / "projector_stage1.pt")
print(f"projector  ({proj.stat().st_size/1e6:.0f} MB)")

# ── FAISS for FROM-SCRATCH Run-4 (RAG index is needed during Stage-2 training) ─
# The resume path (below) restores FAISS from the downloaded dataset; the first
# from-scratch session has no dataset yet, so pull the index from the attached
# vlm-session-state input instead.
if IS_FIRST_SESSION:
    # Clean slate: /kaggle/working persists across sessions, so a prior run can leave
    # models/lora_adapter + cls_head.pt behind. run_pipeline Step 3 SKIPS training when
    # those exist ("Stage 2 up to date"), so from-scratch MUST remove them first or it
    # jumps straight to eval on the OLD model.
    for _d in list(MODELS_DIR.glob("lora_step*")):
        if _d.is_dir(): shutil.rmtree(_d)
    if (MODELS_DIR / "lora_adapter").exists(): shutil.rmtree(MODELS_DIR / "lora_adapter")
    (MODELS_DIR / "cls_head.pt").unlink(missing_ok=True)
    print("from-scratch: wiped stale lora_step*/lora_adapter/cls_head.pt -> Stage 2 trains from step 0")
    fa0 = _walk_find("/kaggle/input", "faiss_index", want_dir=True)
    if fa0:
        dst0 = Path(REPO_DIR) / "faiss_index"
        if dst0.exists(): shutil.rmtree(dst0)
        shutil.copytree(fa0, dst0); print("faiss_index restored from attached input (for from-scratch RAG)")
    else:
        print("WARNING: faiss_index not in attached input - attach vlm-session-state so Stage-2 has the RAG index")

if not IS_FIRST_SESSION:
    # ── Download the LATEST session-state version via API ─────────────────────
    src = None
    try:
        token = secrets.get_secret("KAGGLE_KEY").strip()
        os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
        kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
        if token.startswith("KGAT_"):
            os.environ["KAGGLE_API_TOKEN"] = token
            f = kdir / "access_token"; f.write_text(token); f.chmod(0o600)
        else:
            os.environ["KAGGLE_KEY"] = token
            f = kdir / "kaggle.json"; f.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": token})); f.chmod(0o600)
        DL = Path("/kaggle/working/_state_dl")
        if DL.exists(): shutil.rmtree(DL)
        DL.mkdir(parents=True)
        print(f"downloading latest {KAGGLE_USERNAME}/{STATE_DATASET} ...")
        r = subprocess.run(["kaggle", "datasets", "download", f"{KAGGLE_USERNAME}/{STATE_DATASET}",
                            "--unzip", "-p", str(DL)], capture_output=True, text=True)
        print((r.stdout or "")[-300:]); print((r.stderr or "")[-300:])
        if any(re.match(r"lora_step\d+$", d) for _, dirs, _ in os.walk(DL) for d in dirs):
            src = DL
            print("using freshly-downloaded session-state.")
        else:
            print("download had no checkpoints; falling back to attached input.")
    except Exception as e:
        print(f"API download failed ({e}); falling back to attached input.")

    if src is None:
        src = Path("/kaggle/input")   # fallback to attached version

    # ── WIPE stale on-disk checkpoints (only now that we have a valid source) ──
    for d in list(MODELS_DIR.glob("lora_step*")):
        if d.is_dir(): shutil.rmtree(d)
    if (MODELS_DIR / "lora_adapter").exists():
        shutil.rmtree(MODELS_DIR / "lora_adapter")   # stale final adapter would skip training / break eval
    (MODELS_DIR / "cls_head.pt").unlink(missing_ok=True)
    print("cleared stale on-disk checkpoints + lora_adapter -> will match the dataset exactly")

    # ── FAISS ──────────────────────────────────────────────────────────────────
    fa = _walk_find(src, "faiss_index", want_dir=True)
    if fa:
        dst = Path(REPO_DIR) / "faiss_index"
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(fa, dst); print("faiss_index restored")
    else:
        print("WARNING: faiss_index not found")

    # ── ALL checkpoints from the dataset ───────────────────────────────────────
    pat = re.compile(r"lora_step(\d+)$")
    found = []
    for root, dirs, _f in os.walk(src):
        for d in dirs:
            if pat.match(d): found.append(Path(root) / d)
    if not found:
        raise FileNotFoundError("no lora_step* checkpoint found in dataset/input")
    for s in found:
        dst = MODELS_DIR / s.name
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(s, dst)
    highest = max(int(pat.match(p.name).group(1)) for p in found)
    print(f"checkpoints restored: {sorted(int(pat.match(p.name).group(1)) for p in found)}  (highest={highest})")

    # ── ClassificationHead ──────────────────────────────────────────────────────
    ch = _walk_find(src, "cls_head.pt")
    if ch: shutil.copy2(ch, MODELS_DIR / "cls_head.pt"); print("cls_head restored")
    else:  print("WARNING: cls_head.pt not found")

    # ── Training log ────────────────────────────────────────────────────────────
    lg = _walk_find(src, "stage2.jsonl")
    if lg:
        (Path(REPO_DIR) / "logs").mkdir(exist_ok=True)
        shutil.copy2(lg, Path(REPO_DIR) / "logs" / "stage2.jsonl"); print("log restored")

print("\nAssets ready.")


In [ ]:
# 6. PIPELINE STATE (skip to Stage-2, clear lock) =============================
from pathlib import Path
sp = Path(REPO_DIR) / "logs" / ".pipeline_state"; sp.parent.mkdir(exist_ok=True)
# mark step0,1,2 done so --resume jumps straight to Stage-2 training (step3)
with open(sp, "w") as f:
    for marker in ["step0"]*7 + ["step1", "step2"]:
        f.write(marker + chr(10))
(Path(REPO_DIR) / "logs" / ".pipeline.lock").unlink(missing_ok=True)
print("Pipeline state set (resume into Stage-2). Lock cleared.")


In [ ]:
# 7. PATCH run_pipeline.sh (grad_accum / max_pairs) ===========================
import re
from pathlib import Path
sh = Path(REPO_DIR) / "run_pipeline.sh"; t = sh.read_text()
t = re.sub(r"(GRAD_ACCUM=)\d+",   f"GRAD_ACCUM={GRAD_ACCUM}",  t)
t = re.sub(r"(MAX_PAIRS_S2=)\d+", f"MAX_PAIRS_S2={MAX_PAIRS}", t)
sh.write_text(t)
print(f"patched: GRAD_ACCUM={GRAD_ACCUM}, MAX_PAIRS_S2={MAX_PAIRS}")


In [ ]:
# 8. RUN — TRAIN or EVAL (set MODE in cell 1) =================================
# eval : SWA(last 8) -> calibrate -> evaluate(--tta --rag-ablation --robustness)
#        -> graphs -> upload results to the vlm-eval-results dataset. No training.
# train: resume Stage-2 to 15000, auto-uploading checkpoints to vlm-session-state.
import subprocess, os, shutil, re, json, glob, threading
from pathlib import Path

os.environ["PYTHONUNBUFFERED"] = "1"   # live (unbuffered) child output -> real-time per-sample progress

MODELS = Path(REPO_DIR) / "models"
pat = re.compile(r"lora_step(\d+)$")
steps = sorted(int(pat.match(p.name).group(1)) for p in MODELS.iterdir() if pat.match(p.name))
print(f"checkpoints present: {steps}" if steps else "No checkpoints found!")
print("=" * 60)

def kaggle_auth():
    try:
        token = secrets.get_secret("KAGGLE_KEY").strip()
    except Exception as e:
        print(f"  !! KAGGLE_KEY secret missing ({e}); skipping upload."); return False
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
    if token.startswith("KGAT_"):
        os.environ["KAGGLE_API_TOKEN"] = token
        f = kdir / "access_token"; f.write_text(token); f.chmod(0o600)
    else:
        os.environ["KAGGLE_KEY"] = token
        f = kdir / "kaggle.json"; f.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": token})); f.chmod(0o600)
    return True

def _stream(cmd):
    print("\n>>> " + " ".join(cmd)); print("-" * 60)
    p = subprocess.Popen(cmd, cwd=REPO_DIR, env=os.environ.copy(),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    return p.returncode

def _upload_dir(out_dir, slug, msg):
    if not kaggle_auth():
        print("  no KAGGLE_KEY — results left in", out_dir); return
    (Path(out_dir) / "dataset-metadata.json").write_text(json.dumps(
        {"title": slug, "id": f"{KAGGLE_USERNAME}/{slug}", "licenses": [{"name": "CC0-1.0"}]}, indent=2))
    r = subprocess.run(["kaggle", "datasets", "version", "-p", str(out_dir), "-m", msg, "--dir-mode", "zip"],
                       capture_output=True, text=True)
    blob = (r.stderr or "") + (r.stdout or "")
    if r.returncode != 0 and ("not found" in blob.lower() or "404" in blob):
        r = subprocess.run(["kaggle", "datasets", "create", "-p", str(out_dir), "--dir-mode", "zip"],
                           capture_output=True, text=True)
    print((r.stdout or "")[-400:]); print((r.stderr or "")[-200:])

_upload_lock = threading.Lock()
def package_and_upload(msg="auto-update"):
    """Train-mode: upload EVERY checkpoint (with train_state) to vlm-session-state."""
    with _upload_lock:
        OUT = Path("/kaggle/working/session_state")
        if OUT.exists(): shutil.rmtree(OUT)
        OUT.mkdir(parents=True)
        M = Path(REPO_DIR) / "models"; L = Path(REPO_DIR) / "logs"
        p = re.compile(r"lora_step(\d+)$")
        cps = sorted((int(p.match(x.name).group(1)), x) for x in M.iterdir() if p.match(x.name))
        if not cps:
            print("  no checkpoints yet"); return False
        for _, c in cps:
            dst = OUT / c.name
            if dst.exists(): shutil.rmtree(dst)
            shutil.copytree(c, dst)
        s = M / "cls_head.pt"
        if s.exists(): shutil.copy2(s, OUT / "cls_head.pt")
        fa = Path(REPO_DIR) / "faiss_index"
        if fa.exists(): shutil.copytree(fa, OUT / "faiss_index")
        lg = L / "stage2.jsonl"
        if lg.exists(): shutil.copy2(lg, OUT / "stage2.jsonl")
        print(f"  packaged {len(cps)} checkpoints")
        _upload_dir(OUT, STATE_DATASET, msg); return True

# ════════════════════════════════════════════════════════════════════════════
if MODE == "eval":
    print(">>> EVAL MODE: SWA + calibrate + improved evaluation (no training)\n")
    rc = _stream(["python", "-m", "experiments.average_checkpoints",
                  "--models-dir", "models", "--last-n", "8", "--out", "models/lora_adapter_swa"])
    if rc == 0:
        rc = _stream(["python", "-m", "experiments.calibrate_temperature",
                      "--lora-dir", "models/lora_adapter_swa", "--val-pairs", "500"])
    if rc == 0:
        rc = _stream(["python", "-m", "experiments.evaluate",
                      "--lora-adapter-dir", "models/lora_adapter_swa",
                      "--max-samples", str(EVAL_SAMPLES), "--tta", "--rag-ablation", "--robustness"])
    if rc == 0:
        # FULL ablation study — all 9 experiments (Exp 1-9) on the SWA model
        rc = _stream(["python", "-m", "experiments.run_experiments", "--exp", "all",
                      "--projector", "models/projector_stage1.pt",
                      "--lora-adapter", "models/lora_adapter_swa",
                      "--max-samples", str(EVAL_SAMPLES), "--tgp-w", "55",
                      "--output-dir", "reports/"])
    if rc == 0:
        # single-image qualitative inference -> reports/single_inference.{json,md,png}
        _stream(["python", "-m", "experiments.single_inference",
                 "--image", "sample_xray.jpg",
                 "--lora-adapter-dir", "models/lora_adapter_swa",
                 "--projector-path", "models/projector_stage1.pt",
                 "--output-dir", "reports/"])
    if rc != 0:
        print("\n[eval] a step failed (rc != 0) — see output above; uploading whatever completed.")
    ej = sorted(glob.glob(str(Path(REPO_DIR) / "logs" / "eval_report_*.json")))
    if rc == 0 and ej:
        _stream(["python", "-m", "experiments.visualize", "--eval-json", ej[-1],
                 "--stage2-log", "logs/stage2.jsonl", "--out-dir", "diagnostics/", "--format", "png"])
    # stage + upload eval results (small) to vlm-eval-results
    OUT = Path("/kaggle/working/eval_results")
    if OUT.exists(): shutil.rmtree(OUT)
    OUT.mkdir(parents=True)
    if ej: shutil.copy2(ej[-1], OUT / Path(ej[-1]).name)
    diag = Path(REPO_DIR) / "diagnostics"
    if diag.exists() and any(diag.iterdir()): shutil.copytree(diag, OUT / "diagnostics")
    rep = Path(REPO_DIR) / "reports"
    if rep.exists() and any(rep.iterdir()): shutil.copytree(rep, OUT / "reports")   # Exp 1-9 results
    swa = Path(REPO_DIR) / "models" / "lora_adapter_swa"
    if swa.exists(): shutil.copytree(swa, OUT / "lora_adapter_swa")
    chp = Path(REPO_DIR) / "models" / "cls_head.pt"
    if chp.exists(): shutil.copy2(chp, OUT / "cls_head.pt")
    print("\n[eval] uploading results to vlm-eval-results ...")
    _upload_dir(OUT, "vlm-eval-results", "eval: SWA + calibrate + TTA")
    print("\nEVAL COMPLETE -> results in the vlm-eval-results dataset + /kaggle/working/eval_results/")
else:
    if steps:
        ok = (MODELS / f"lora_step{steps[-1]}" / "train_state.pt").exists()
        print(f"RESUME TARGET: lora_step{steps[-1]}  (train_state.pt: {'OK' if ok else 'MISSING!'})")
    _stop = threading.Event()
    def _periodic():
        last = -1
        while not _stop.wait(1800):
            cur = max([int(pat.match(p.name).group(1)) for p in MODELS.iterdir() if pat.match(p.name)] or [-1])
            if cur > last:
                print(f"\n[uploader] new checkpoint {cur} -> uploading all ...")
                if package_and_upload(f"autosave through step {cur}"): last = cur
    threading.Thread(target=_periodic, daemon=True).start()
    print("[uploader] auto-upload ON\n")
    proc = subprocess.Popen(["bash", "run_pipeline.sh", "--resume"], cwd=REPO_DIR,
                            env=os.environ.copy(), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        proc.terminate(); print("\n[notebook] interrupted - saving now...")
    proc.wait(); _stop.set()
    print(f"\n[notebook] exit code {proc.returncode}")
    package_and_upload("final - all checkpoints")
    print("Download: https://www.kaggle.com/datasets/" + KAGGLE_USERNAME + "/" + STATE_DATASET)


In [ ]:
# 9. FORCE UPLOAD NOW (optional) — pushes ALL checkpoints immediately ==========
# Run any time you want to snapshot the full set right now (does not stop training
# only if cell 8 has finished/been interrupted, since the kernel is otherwise busy).
package_and_upload("manual full snapshot")


## Downloading the weights

You normally never need to — cell 8 pushes everything to `vlm-session-state`,
and the next session pulls it automatically.

To pull weights to your laptop (stable, unlike the in-session Output tab):
- **Dataset page**: https://www.kaggle.com/datasets/sujalprasad/vlm-session-state -> three-dot menu -> Download
- **CLI**:
  ```
  export KAGGLE_API_TOKEN=KGAT_xxx
  kaggle datasets download sujalprasad/vlm-session-state --unzip -p ./out
  ```

## What was fixed vs the old notebook
- No pruning anywhere — every checkpoint kept and uploaded.
- Resume verified before training (cell 8 prints the exact target step).
- `git reset --hard` before pull — no more merge conflicts.
- Auto-discover dataset paths — mount path can't break it.
- Auto-upload via `KGAT_` token — downloads actually work.
- Kermany pre-load capped at 400 (in repo) — no RAM OOM.
